# **Градиент-free CAM: Score-CAM, Ablation-CAM, Eigen-CAM**

Практика к модулю [«Атрибуция: локализация и CAM-семейство»](https://open-xai-platform.web.app).

В уроке мы разобрали три способа получить веса карт признаков, не спрашивая градиент,
и сказали, чем каждый платит. Здесь всё это считается руками — поверх той же ResNet-50,
что и в практике по Grad-CAM.

К концу тетради у вас будет:

- свои реализации Score-CAM, Ablation-CAM и Eigen-CAM (по десятку строк каждая);
- измеренная цена отказа от градиента — в прямых проходах, а не на словах;
- проверка того, что Eigen-CAM **не зависит от класса**, — на одной картинке с двумя объектами.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import requests
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

## Модель, картинка и общая часть

Все методы семейства устроены одинаково: карта — это взвешенная сумма карт признаков
$\sum_k w^c_k A^k$, и различаются они только тем, откуда берутся веса. Значит общая часть —
получить карты признаков последнего свёрточного слоя.

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval();

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# На этой фотографии два объекта разных классов — она понадобится в конце.
image = Image.open(BytesIO(requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat_and_dog.jpg').content)).convert('RGB')
x = transform(image).unsqueeze(0)

categories = [s.strip() for s in requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/imagenet_classes.txt').text.splitlines()]
with torch.no_grad():
    logits = model(x)
top = torch.topk(logits, 3).indices[0].tolist()
print('топ-3:', [(i, categories[i]) for i in top])

In [ ]:
def feature_maps(model, x, layer):
    """Карты признаков указанного слоя: тензор (C, h, w)."""
    store = {}
    handle = layer.register_forward_hook(lambda m, i, o: store.__setitem__('a', o))
    with torch.no_grad():
        model(x)
    handle.remove()
    return store['a'][0]

target_layer = model.layer4[-1]
A = feature_maps(model, x, target_layer)
print('карт признаков:', A.shape[0], '· размер каждой:', tuple(A.shape[1:]))

## 1. Score-CAM: спросить у модели напрямую

Вес канала — это логит целевого класса на изображении, замаскированном картой этого канала.
Никакого обратного прохода: только прямые.

$$w^c_k = f_c(x \odot \text{norm}(A^k)) - f_c(\text{пустое изображение})$$

In [ ]:
@torch.no_grad()
def score_cam(model, x, layer, cls, batch=64):
    A = feature_maps(model, x, layer)
    C = A.shape[0]

    # растягиваем карты до размера входа и нормируем в [0, 1] — получаются маски
    M = F.interpolate(A.unsqueeze(0), x.shape[-2:], mode='bilinear', align_corners=False)[0]
    lo = M.flatten(1).min(1).values[:, None, None]
    hi = M.flatten(1).max(1).values[:, None, None]
    M = (M - lo) / (hi - lo + 1e-8)

    base = model(torch.zeros_like(x))[0, cls]      # логит на пустом изображении
    weights, passes = [], 1
    for i in range(0, C, batch):
        masked = x * M[i:i + batch].unsqueeze(1)
        weights.append(model(masked)[:, cls] - base)
        passes += masked.shape[0]
    w = torch.cat(weights)

    cam = torch.relu((w[:, None, None] * A).sum(0))
    return cam / (cam.max() + 1e-8), passes

cls = top[0]
sc_cam, sc_passes = score_cam(model, x, target_layer, cls)
print(f'Score-CAM: прямых проходов {sc_passes}')

**Задание 1.** Сколько прямых проходов потребует Score-CAM, если считать его
на слое `layer3` вместо `layer4`? Посчитайте, не запуская: число каналов слоя видно
в `print(model)`.

In [ ]:
# Ваш код здесь

## 2. Ablation-CAM: выключить и посмотреть

Зеркальная мысль: вес канала — это то, насколько просядет логит, если канал занулить.

$$w^c_k = \frac{y^c - y^c_{\setminus k}}{y^c}$$

Считать всю сеть заново не нужно — достаточно хвоста после интересующего слоя.
Здесь мы для наглядности зануляем канал прямо в картах признаков и прогоняем остаток сети.

In [ ]:
@torch.no_grad()
def ablation_cam(model, x, layer, cls):
    A = feature_maps(model, x, layer)
    C = A.shape[0]

    def head(maps):
        """Хвост ResNet после layer4: пулинг и классификатор."""
        v = model.avgpool(maps.unsqueeze(0)).flatten(1)
        return model.fc(v)[0, cls]

    y = head(A)
    weights = []
    for k in range(C):
        ablated = A.clone()
        ablated[k] = 0
        weights.append((y - head(ablated)) / (y + 1e-8))
    w = torch.stack(weights)

    cam = torch.relu((w[:, None, None] * A).sum(0))
    return cam / (cam.max() + 1e-8)

ab_cam = ablation_cam(model, x, target_layer, cls)
print('Ablation-CAM посчитан')

**Задание 2.** В уроке сказано: когда несколько каналов дублируют друг друга,
Ablation-CAM занижает важность каждого — выключишь один, соседи компенсируют. Найдите
в векторе весов долю каналов, у которых вес по модулю меньше 0.001. Что это за каналы?

In [ ]:
# Ваш код здесь

## 3. Eigen-CAM: вообще без класса

Раскладываем активации в матрицу «позиции × каналы» и берём первую главную компоненту.
Класс здесь не участвует вовсе — и это главное свойство метода.

In [ ]:
@torch.no_grad()
def eigen_cam(model, x, layer):
    A = feature_maps(model, x, layer)
    C, h, w = A.shape
    flat = A.reshape(C, h * w).T                    # (позиции, каналы)
    flat = flat - flat.mean(0, keepdim=True)
    _, _, V = torch.linalg.svd(flat, full_matrices=False)
    cam = (flat @ V[0]).reshape(h, w)               # проекция на первую компоненту
    cam = torch.relu(cam)
    return cam / (cam.max() + 1e-8)

ei_cam = eigen_cam(model, x, target_layer)
print('Eigen-CAM посчитан')

## 4. Главная проверка: зависит ли карта от класса

На фотографии два объекта. Строим карты для двух разных классов и смотрим, изменилось ли
хоть что-нибудь. У честного к классу метода карты обязаны разойтись.

In [ ]:
# Берём два класса разной природы: собачий из топ-1 и кошачий — на фото есть и тот и другой.
cls_a = top[0]
cls_b = max(range(len(categories)),
            key=lambda i: logits[0, i].item() if 'cat' in categories[i] else -1e9)
print('класс A:', categories[cls_a], '· класс B:', categories[cls_b])

def diff(m1, m2):
    """Средняя разница двух карт, приведённых к [0, 1]."""
    return (m1 - m2).abs().mean().item()

sc_a, _ = score_cam(model, x, target_layer, cls_a)
sc_b, _ = score_cam(model, x, target_layer, cls_b)
ab_a = ablation_cam(model, x, target_layer, cls_a)
ab_b = ablation_cam(model, x, target_layer, cls_b)
ei = eigen_cam(model, x, target_layer)

print(f'Score-CAM    A против B: {diff(sc_a, sc_b):.4f}')
print(f'Ablation-CAM A против B: {diff(ab_a, ab_b):.4f}')
print(f'Eigen-CAM    A против B: {diff(ei, ei):.4f}   ← класс в метод не входит вовсе')

**Задание 3.** Чему равна средняя разница карт Score-CAM для двух классов
(округлите до сотых)? А у Eigen-CAM — и почему именно столько?

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(image.resize((224, 224))); ax[0].set_title('оригинал')
for a, (m, t) in zip(ax[1:], [(sc_a, 'Score-CAM'), (ab_a, 'Ablation-CAM'), (ei, 'Eigen-CAM')]):
    a.imshow(image.resize((224, 224)))
    a.imshow(F.interpolate(m[None, None], (224, 224), mode='bilinear')[0, 0].numpy(),
             cmap='jet', alpha=0.5)
    a.set_title(t)
for a in ax:
    a.axis('off')
plt.tight_layout()
plt.show()

## Что унести из тетради

| Метод | Чем платит | Когда брать |
| --- | --- | --- |
| Score-CAM | прямой проход на каждый канал | градиенты шумят, а время есть |
| Ablation-CAM | то же, но считается только хвост сети | нужна важность «в контексте» остальных каналов |
| Eigen-CAM | **не зависит от класса** | быстрая отладка, задачи без логитов |

И общее, что не меняется от смены способа получения весов: все три метода живут
в разрешении карт признаков ($7\times7$ для входа $224\times224$) и отвечают на вопрос
«в какой области», а не «по каким пикселям».